# **Práctico 7** 
**Entrada/Salida**

----

## Resumen práctico

Los programas muchas veces necesitan:

- Leer información desde archivos.
- Guardar resultados en archivos.
- Acceder a información externa (por ejemplo páginas web o APIs).

Python permite trabajar principalmente con:

- Archivos de texto (`.txt`, `.csv`, `.py` etc.)
- Archivos binarios (imágenes, videos, ejecutables, etc.)

## Tratamiento de archivos
La función principal es:

```python
open(nombre_archivo, modo)
```
abre un archivo y devuelve un objeto de archivo ("file handle"), sus modos más usados

| Modo | Significado |
|---|---|
| `"r"` | lectura |
| `"w"` | escritura |
| `"a"` | agregar contenido |
| `"x"`| creacion exclusiva (da un error si el archivo ya existe) |

Solo en 'r' da error si el archivo no existe `FileNotFoundError`, en el resto de los modos si no exite lo crea. 


- Para cear y escribir un archivo



In [9]:
archivo = open("datos.txt", "w")

archivo.write("Hola\n")
archivo.write("Python\n")

archivo.close()

**Importante**

- `"w"` borra el contenido previo si el archivo ya existía.
- `write()` escribe texto.
- `close()` cierra el archivo.

En el proceso típico al trabajar con archivos siempre hay 3 etapas:

1. Abrir el archivo
2. Procesarlo
3. Cerrarlo

Cerrar el archivo es importante para evitar errores y liberar recursos.

- Para leer un archivo se usa  `read()` 

Su output es un `str`. Por defecto, lee el archivo completo desde la posición actual del cursor hasta el final. Mueve el cursor, es decir al terminar de leer, el "puntero" del archivo se queda al final. Si llamas a `read()` de nuevo inmediatamente, devolverá una cadena vacía. 

Solo acepta un parámetro opcional: `size` (número entero). Que indica la cantidad máxima de caracteres (en modo texto) o bytes (en modo binario) que deseas leer. Si no se pone (o es negativo): Lee todo el contenido hasta el final del archivo. Si se pone: Lee solo hasta ese número de caracteres.



In [10]:
archivo = open("datos.txt", "r")

contenido = archivo.read()

print(contenido)

archivo.close()

Hola
Python



Cada llamada a `read()` continúa desde donde quedó la anterior.


In [11]:
archivo = open("datos.txt", "r")

a = archivo.read(3)
b = archivo.read(4)

print(a)
print(b)

archivo.close()

Hol
a
Py


Usando `readline()` puedo leer el contenido de un archivo línea por línea. Es ideal para procesar archivos grandes sin saturar la memoria RAM. Empieza desde donde esté el cursor y se detiene al encontrar un salto de línea (`\n`) o el final del archivo. El resultado devuelto incluye el carácter `\n` al final, a menos que sea la última línea del archivo y no lo tenga. Cada vez que lo llamas, el cursor se mueve a la siguiente línea.

In [12]:
archivo = open("datos.txt", "r")

linea = archivo.readline()
while linea != "":
    print(linea, end="")
    linea = archivo.readline()

archivo.close()

Hola
Python


In [13]:
archivo = open("datos.txt", "r")

for linea in archivo:
    print(linea, end="")

archivo.close()

Hola
Python


- Para agregar información a un archivo uso el modo append (`"a"`). No borra el contenido previo y agrega al final del archivo.

In [14]:
archivo = open("datos.txt", "a")

archivo.write("Nueva línea\n")

archivo.close()

## Programación defensiva
Cuando trabajamos con archivos pueden ocurrir errores:

- el archivo no existe
- no tenemos permisos
- el archivo está corrupto

Por eso usamos `try` y `except`, haciendo manejo de excepciones. La estructura básica

```python
try:
    # se intenta ejecutar esto

except:
    # si hay error, entra acá
else:
    # (Opcional) Se ejecuta solo si no hubo errores en el try.
finally:
    # (Opcional) se ejecuta siempre. Es ideal para cerrar archivos o liberar
```

El orden de ejecución es el siguiente
1. La falla: Ocurre un error dentro del bloque try.
2. La búsqueda: Python detiene el try y mira el primer except.
3. La coincidencia:Si el error coincide, ejecuta ese bloque y salta todos los demás except. Dato, se pueden poner varios errores ejemplo `except (ValueError, TypeError):`
4. Si no coincide, pasa al siguiente except.
5. La caída final: Si ningún except coincide, el programa truena (a menos que tengas un except genérico al final, por ejemplo `except Exception:`).

Por ejemplo

In [19]:
try:
    archivo = open("inexistente.txt", "r")

    contenido = archivo.read()

except FileNotFoundError:
    print("El archivo no existe")

finally:
    archivo.close()


El archivo no existe


Errrores mas comunes al trabajar con archivos:

- `FileNotFoundError`: Intentas abrir un archivo que no existe.
- `ValueError`: Intentas convertir algo a un tipo de dato que no cuadra (ej. int("hola")).
- `TypeError`: Haces una operación con tipos incompatibles (ej. sumar un número y un texto).
- `IndexError`: Intentas acceder a un índice de una lista que está fuera de rango.
- `ZeroDivisionError`: Dividir cualquier número por cero.

Otros comunes:

- `SyntaxError`: El código está mal escrito. Te falta un paréntesis, un dos puntos (:) al final de un if, o tienes una comilla sin cerrar. 
- `IndentationError`: Un clásico de Python. Ocurre cuando los espacios o pestañas (la sangría) no están alineados correctamente.
- `NameError`: Intentas usar una variable o llamar a una función que no ha sido definida antes. Suele pasar por errores de dedo (ej. escribes nobre en lugar de nombre).
- `KeyError`: Intentas buscar una clave en un diccionario que no existe.
- `AttributeError`: Intentas llamar a un método en un objeto que no lo tiene. 
- `ImportError / ModuleNotFoundError`: Intentas importar una librería que no está instalada o escribiste mal el nombre del módulo.

## Verificar si un archivo existe

**Opción 1 — usando try**

```python
def existe_archivo(nombre):
    try:
        archivo = open(nombre, "r")
        archivo.close()
        return True

    except IOError:
        return False
```
**Opción 2 — usando `os.path`**

```python
import os.path

if os.path.isfile("datos.txt"):
    print("Existe")
else:
    print("No existe")
```

**Opción 3 - usando  de `with`**

La forma que recomendamos.

```python
with open("datos.txt", "r") as archivo:
    contenido = archivo.read()

print(archivo.closed)
```

- cierra el archivo automáticamente
- evita olvidarse de `close()`
- funciona bien incluso si hay errores

## Manejo de directorios
Dado que trabajaremos con archivos, es importante tener un buen manejo de rutas.

- Para obtener directorio actual

```python
import os

print(os.getcwd())
```
- Listar archivos

```python
import os

for nombre in os.scandir("."):
    print(nombre)
```

- Crear y eliminar directorios

    - Crear carpeta

    ```python
    import os

    os.mkdir("mi_carpeta")
    ```
    - Eliminar carpeta vacía

    ```python
    os.rmdir("mi_carpeta")
    ```
    - Eliminar archivo

    ```python
    os.remove("datos.txt")
    ```




## Lectura de datos desde internet

Python puede acceder a sitios web usando la librería `requests`

- Hacer una petición `GET`

```python
import requests

respuesta = requests.get("https://api.github.com")

print(respuesta.status_code)
```

HTTP y  sus códigos de estado

| Código | Significado |
|---|---|
| 200 | éxito |
| 404 | no encontrado |
| 500 | error del servidor |
| 503 | servicio no disponible |

La libreria tiene su propio manejo de errores 

```python
import requests

try:
    respuesta = requests.get("https://sitio.com")

except requests.exceptions.RequestException:
    print("Error en la conexión")
```



### Ejercicio 1
Implementar la función:
```python
def crear_archivo(nombre: str) -> None:
    ...
```
La función debe crear un archivo de texto con el nombre recibido por parámetro y escribir exactamente las siguientes líneas:
```python
Hola
Bienvenidos a Algoritmos y Programación
Python es divertido
```
Considerar que cada frase debe quedar en una línea distinta dentro del archivo.

In [ ]:
# codigo

### Ejercicio 2

Implementar la función:
```python
def leer_archivo(nombre: str) -> str:
    ...
```
La función debe:

- abrir el archivo en modo lectura
- leer todo su contenido
- devolver dicho contenido como un `str`

Si el archivo no existe, la función debe devolver:
```python
"ERROR"
```

In [ ]:
# codigo

### Ejercicio 3
Implementar la función:
```python
def contar_lineas(nombre: str) -> int:
    ...
```
La función debe devolver la cantidad total de líneas presentes en un archivo de texto.

Si el archivo no existe, la función debe devolver `-1`

In [ ]:
#codigo

### Ejercicio 4

**1.** Implementar la función:
```python
def contar_caracteres(nombre: str) -> int:
    ...
```
La función debe devolver la cantidad total de caracteres contenidos en el archivo Considerar también los espacios y saltos de línea.


**2.** Implementar la función:
```python
def contar_palabras(nombre: str) -> int:
    ...
```
La función debe devolver la cantidad total de palabras presentes en el archivo. Suponer que las palabras están separadas por espacios.

In [ ]:
# codigo

### Ejercicio 5


Implementar la función:

```python
def copiar_archivo(origen: str, destino: str) -> None:
    ...
```
La función debe leer todo el contenido del archivo origen y copiarlo en un nuevo archivo llamado destino.

Para resolver este ejercicio, se debe utilizar:

- `with` para abrir y cerrar los archivos correctamente.
- `try` y `except` para manejar posibles errores.

Si el archivo origen no existe, la función debe imprimir:
```python
Archivo inexistente
```

In [ ]:
# codigo

### Ejercicio 6

Implementar la función:
```python
def agregar_linea(nombre: str, linea: str) -> None:
    ...
```

La función debe agregar el contenido recibido en linea al final del archivo indicado por nombre. Para resolver este ejercicio, se debe utilizar:

- `with` para abrir el archivo.
- El modo de apertura adecuado para no borrar el contenido previo.
- `try` y `except` para manejar posibles errores al trabajar con el archivo.

In [ ]:
# codigo

### Ejercicio 7
Implementar la función:
```python
def imprimir_lineas_numeradas(nombre: str) -> None:
    ...
```

La función debe leer el archivo indicado por nombre e imprimir cada línea precedida por su número de línea.

Por ejemplo, si el archivo contiene:
```python
primera línea
segunda línea
tercera línea
```
la salida esperada es:
```python
1: primera línea
2: segunda línea
3: tercera línea
```

Para resolver este ejercicio, se debe utilizar:

- `with` para abrir el archivo.
- `try` y `except` para manejar el caso en que el archivo no exista.

Si el archivo no existe, imprimir:
```python
Archivo inexistente
```

In [ ]:
# codigo

### Ejercicio 8
Implementar la función:
```python
def buscar_palabra(nombre: str, palabra: str) -> bool:
    ...
```
La función debe devolver True si la palabra aparece en el contenido del archivo indicado por nombre, y False en caso contrario.

Para resolver este ejercicio, se debe utilizar:

- `with` para abrir el archivo.
- `try` y `except` para manejar posibles errores.

Si el archivo no existe, la función debe devolver `False`.

In [ ]:
# codigo

### Ejercicio 9
Implementar la función:
```python
def contar_vocales(nombre: str) -> dict:
    ...
```
La función debe leer el contenido del archivo indicado por nombre y devolver un diccionario con la cantidad de apariciones de cada vocal minúscula:
```python
{
    "a": 0,
    "e": 0,
    "i": 0,
    "o": 0,
    "u": 0
}
```

Por ejemplo, si el archivo contiene `Hola mundo` la función debe devolver:
```python
{
    "a": 1,
    "e": 0,
    "i": 0,
    "o": 2,
    "u": 1
}
```

Para resolver este ejercicio, se debe utilizar:

- `with` para abrir el archivo.
- `try` y `except` para manejar posibles errores.
- Un diccionario para acumular las apariciones de cada vocal.

Si el archivo no existe, la función debe devolver el diccionario con todas las vocales en 0.

In [ ]:
# codigo

### Ejercicio 10


Implementar la función:

```python
def frecuencia_palabras(nombre_archivo: str) -> dict[str, int]:
    ...
```

La función debe leer un archivo de texto y devolver un diccionario donde:

- las claves sean las palabras que aparecen en el archivo
- los valores sean la cantidad de veces que aparece cada palabra



La función debe:

1. abrir el archivo en modo lectura
2. leer todo su contenido
3. convertir el texto a minúsculas
4. eliminar signos de puntuación como:

```text
.,;:!?()[]{}"'
```

5. separar el texto en palabras
6. construir un diccionario de frecuencias

Ejemplo de uso. Si el archivo contiene:

```text
Hola hola mundo.
Python, mundo!
```

La función debe devolver:

```python
{
    "hola": 2,
    "mundo": 2,
    "python": 1
}
```

Restricciones

- No se permite usar `collections.Counter`
- Debe resolverse utilizando diccionarios
- Utilizar `with open(...)`
- Si el archivo no existe, devolver:

    ```python
    {}
    ```

In [ ]:
# codigo

### Ejercicio 11

**1.** Implementar la función:

```python
def palabras_unicas(nombre_archivo: str) -> set[str]:
    ...
```

La función debe devolver un conjunto con todas las palabras únicas
que aparecen en un archivo de texto.


La función debe:

1. abrir el archivo en modo lectura
2. leer el contenido completo
3. convertir todas las palabras a minúsculas
4. eliminar signos de puntuación como:

```text
.,;:!?()[]{}"'
```

5. separar el texto en palabras
6. construir y devolver un conjunto (`set`) con las palabras distintas

Ejemplo de uso. Si el archivo contiene:

```text
Hola mundo.
Hola Python.
```

La función debe devolver:

```python
{"hola", "mundo", "python"}
```

Restricciones

- Debe utilizarse un conjunto (`set`)
- Utilizar `with open(...)`
- No se permite usar librerías externas
- Si el archivo no existe, devolver:

    ```python
    set()
    ```

**2.**

Implementar la función:

```python
def palabras_en_comun(archivo1: str, archivo2: str) -> set[str]:
    ...
```

La función debe devolver un conjunto con las palabras que aparecen
en ambos archivos.

La función debe:

1. obtener las palabras únicas del primer archivo
2. obtener las palabras únicas del segundo archivo
3. calcular las palabras compartidas entre ambos conjuntos
4. devolver el resultado

Ejemplo de uso 
- archivo1.txt:

    ```text
    Hola mundo Python
    ```
- archivo2.txt:

    ```text
    Python es divertido mundo
    ```

Resultado esperado:

```python
{"python", "mundo"}
```

Restricciones

- Resolver utilizando operaciones entre conjuntos (`set`)
- Si alguno de los archivos no existe, devolver:

    ```python
    set()
    ```

In [ ]:
# codigo

### Ejercicio 12


Un archivo `alumnos.txt` contiene información de estudiantes (creenlo para el ejercicio). Cada línea del archivo tiene el siguiente formato:

```text
nombre,nota1,nota2,nota3
```

Ejemplo:

```text
Juan,8,7,10
Ana,4,5,6
Pedro,9,9,8
```

**1.** Implementar la función:

```python
def cargar_alumnos(nombre_archivo: str) -> list[tuple[str, float]]:
    ...
```

La función debe devolver una lista de tuplas con la forma:

```python
(nombre, promedio)
```

donde:

- `nombre` es el nombre del alumno
- `promedio` es el promedio de sus tres notas


La función debe:

1. abrir el archivo en modo lectura
2. leer el archivo línea por línea
3. separar cada línea utilizando `split(",")`
4. convertir las notas a números
5. calcular el promedio de cada alumno
6. guardar el resultado en una lista de tuplas

Ejemplo de uso:

Para el archivo:

```text
Juan,8,7,10
Ana,4,5,6
```

La función debe devolver:

```python
[
    ("Juan", 8.333333333333334),
    ("Ana", 5.0)
]
```

Restricciones:
- Utilizar `with open(...)`
- Resolver utilizando listas y tuplas
- No utilizar librerías externas
- Si el archivo no existe, devolver:

    ```python
    []
    ```

**2.**  Implementar la función:

```python
def guardar_reporte(alumnos: list[tuple[str, float]], nombre_archivo: str) -> None:
    ...
```

La función debe generar un archivo de reporte con los alumnos ordenados
de mayor a menor promedio.

Además, debe indicar si cada alumno:

- `"APROBADO"` si el promedio es mayor o igual a 6
- `"DESAPROBADO"` en caso contrario


La función debe:

1. ordenar la lista de alumnos por promedio de mayor a menor
2. crear el archivo de salida
3. escribir una línea por alumno

Formato esperado:

```text
Juan - Promedio: 8.33 - APROBADO
Pedro - Promedio: 7.00 - APROBADO
Ana - Promedio: 5.00 - DESAPROBADO
```


- Utilizar `with open(...)`
- Utilizar `sorted(...)`
- El promedio debe mostrarse con 2 decimales


In [ ]:
# codigo

### EJercicio 13
Vamos a crear un sistema de inventario con persistencia en archivos

Implementar una clase:

```python
class Inventario:
    ...
```

La clase debe administrar productos utilizando un diccionario interno
con el siguiente formato:

```python
{
    "producto": cantidad
}
```

Ejemplo:

```python
{
    "lapiz": 10,
    "cuaderno": 5,
    "goma": 2
}
```

Requisitos generales

- El inventario debe almacenarse en un atributo de instancia
- Utilizar diccionarios para guardar los productos
- Utilizar manejo de excepciones cuando corresponda
- Utilizar `with open(...)` para trabajar con archivos


**1.**  Constructor

La clase debe inicializar un inventario vacío.

Ejemplo:

```python
inv = Inventario()
```

Estado interno esperado:

```python
{}
```

**2.** Agregar productos

Implementar el método:

```python
def agregar(producto: str, cantidad: int) -> None:
    ...
```

El método debe:

- agregar el producto al inventario si no existe
- sumar la cantidad al stock existente si el producto ya está cargado

Ejemplo:

```python
inv.agregar("lapiz", 10)
inv.agregar("lapiz", 5)
```

Resultado esperado:

```python
{
    "lapiz": 15
}
```

Validaciones:

- `producto` debe ser un `str`
- `cantidad` debe ser un entero positivo

En caso contrario, lanzar una excepción adecuada.

**3.** Quitar productos

Implementar el método:

```python
def quitar(producto: str, cantidad: int) -> None:
    ...
```

El método debe descontar unidades del producto indicado.

Ejemplo:

```python
inv.quitar("lapiz", 3)
```

Si había:

```python
{
    "lapiz": 10
}
```

Debe quedar:

```python
{
    "lapiz": 7
}
```
Validaciones

El método debe lanzar excepciones si:

- el producto no existe
- la cantidad es inválida
- no hay suficiente stock

Ejemplo esperado:

```python
raise ValueError("Stock insuficiente")
```

**4.** Guardar inventario

Implementar el método:

```python
def guardar(nombre_archivo: str) -> None:
    ...
```

El método debe guardar el inventario en un archivo de texto.

Formato esperado del archivo:

```text
lapiz,10
cuaderno,5
goma,2
```

Cada línea debe contener:

```text
producto,cantidad
```

Requisitos

- Utilizar `with open(...)`
- Abrir el archivo en modo escritura
- Sobrescribir el contenido previo

**5.** Cargar inventario

Implementar el método:

```python
def cargar(nombre_archivo: str) -> None:
    ...
```

El método debe leer un archivo y reconstruir el inventario.


La función debe:

1. abrir el archivo en modo lectura
2. leer línea por línea
3. separar cada línea utilizando `split(",")`
4. convertir las cantidades a enteros
5. reconstruir el diccionario interno

Caso de error

Si el archivo no existe:

- lanzar una excepción
o
- informar el error mediante `try/except`

(según el diseño elegido)

Ejemplo de uso completo, donde el resultado esperado es:

```python
{
    "lapiz": 10,
    "goma": 5
}
```


In [ ]:
inv = Inventario()

inv.agregar("lapiz", 10)
inv.agregar("goma", 5)

inv.guardar("inventario.txt")

nuevo = Inventario()
nuevo.cargar("inventario.txt")

print(nuevo._productos)